# Agente de Viajes con Human-in-the-Loop usando Microsoft Agent Framework

## Descripción

En este ejercicio, crearás un **agente de reservas de viajes corporativos** que implementa el patrón **Human-in-the-Loop (HITL)**. El agente recomendará opciones de viaje, pero requerirá aprobación humana antes de realizar reservas costosas o fuera de política.

### ¿Qué es Human-in-the-Loop?

Human-in-the-Loop es un patrón de diseño donde los sistemas de IA trabajan junto con humanos para tomar decisiones críticas. En lugar de depender únicamente de decisiones automatizadas, HITL garantiza que el juicio humano y la inteligencia de la IA se complementen.

### Casos de uso:
- Aprobaciones de transacciones financieras
- Validación de decisiones médicas
- Autorización de reservas de alto costo
- Verificación de cambios de configuración críticos

### Cargar librerías

Importamos las bibliotecas necesarias para construir nuestro agente con capacidad de aprobación humana.

In [ ]:
import os
import asyncio
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIChatClient
from agent_framework import ai_function
from pydantic import Field
from typing import Annotated

# GitHub Models client configuration
MODEL_NAME = os.getenv("GITHUB_MODEL", "openai/gpt-4o")

print("✓ Bibliotecas cargadas exitosamente")
print(f"✓ Modelo configurado: {MODEL_NAME}")

### Definir funciones con aprobación requerida

Creamos funciones que el agente puede usar, pero marcamos algunas con `approval_mode="always_require"` para que requieran aprobación humana antes de ejecutarse.

**Funciones del agente:**
1. `search_flights` - Buscar vuelos disponibles (sin aprobación)
2. `book_flight` - Reservar vuelo (requiere aprobación)
3. `book_hotel` - Reservar hotel (requiere aprobación)

In [ ]:
# Función sin aprobación - búsqueda de información
@ai_function
def search_flights(
    origin: Annotated[str, Field(description="Ciudad de origen")],
    destination: Annotated[str, Field(description="Ciudad de destino")],
    date: Annotated[str, Field(description="Fecha del viaje (YYYY-MM-DD)")]
) -> str:
    """
    Busca vuelos disponibles entre dos ciudades.
    Esta función NO requiere aprobación humana.
    """
    # Simulación de búsqueda de vuelos
    flights = [
        {"flight": "IB2034", "price": "450 EUR", "departure": "08:00", "arrival": "11:30"},
        {"flight": "VY7891", "price": "320 EUR", "departure": "14:00", "arrival": "17:20"},
        {"flight": "UX5612", "price": "580 EUR", "departure": "19:00", "arrival": "22:15"}
    ]
    
    result = f"\n🔍 Vuelos encontrados de {origin} a {destination} para {date}:\n"
    for flight in flights:
        result += f"  - {flight['flight']}: {flight['price']} | Salida: {flight['departure']} | Llegada: {flight['arrival']}\n"
    
    return result


# Función CON aprobación - acción crítica
@ai_function(approval_mode="always_require")
def book_flight(
    flight_number: Annotated[str, Field(description="Número de vuelo")],
    passenger_name: Annotated[str, Field(description="Nombre del pasajero")],
    price: Annotated[str, Field(description="Precio del vuelo")]
) -> str:
    """
    Reserva un vuelo específico.
    Esta función REQUIERE aprobación humana antes de ejecutarse.
    """
    return f"✈️ Vuelo {flight_number} reservado exitosamente para {passenger_name}. Precio: {price}. Confirmación enviada por email."


# Función CON aprobación - acción crítica
@ai_function(approval_mode="always_require")
def book_hotel(
    hotel_name: Annotated[str, Field(description="Nombre del hotel")],
    city: Annotated[str, Field(description="Ciudad")],
    check_in: Annotated[str, Field(description="Fecha de check-in (YYYY-MM-DD)")],
    nights: Annotated[int, Field(description="Número de noches")],
    price: Annotated[str, Field(description="Precio total")]
) -> str:
    """
    Reserva una habitación de hotel.
    Esta función REQUIERE aprobación humana antes de ejecutarse.
    """
    check_out_msg = f"Check-in: {check_in}, {nights} noches"
    return f"🏨 Reserva confirmada en {hotel_name}, {city}. {check_out_msg}. Total: {price}. Confirmación enviada."


print("✓ Funciones de agente definidas:")
print("  - search_flights (sin aprobación)")
print("  - book_flight (CON aprobación humana)")
print("  - book_hotel (CON aprobación humana)")

### Crear el agente de viajes con HITL

Ahora creamos un agente que puede usar estas funciones. Cuando el agente intente ejecutar una función que requiere aprobación, el flujo se detendrá y esperará la decisión humana.

In [ ]:
async def run_travel_agent_with_hitl():
    """
    Ejecuta el agente de viajes con Human-in-the-Loop.
    Demuestra cómo manejar aprobaciones humanas en el flujo del agente.
    """
    
    print("\n" + "="*70)
    print("🤖 AGENTE DE VIAJES CON HUMAN-IN-THE-LOOP")
    print("="*70 + "\n")
    
    # Configurar cliente
    client = OpenAIChatClient(
        model_id=MODEL_NAME,
        api_key=os.environ["GITHUB_TOKEN"],
        base_url="https://models.github.ai/inference"
    )
    
    # Crear agente con las herramientas
    async with ChatAgent(
        chat_client=client,
        name="travel_agent",
        instructions="""Eres un agente de viajes corporativos profesional.
        
        Tu trabajo es ayudar a los empleados a encontrar y reservar viajes de negocios.
        
        IMPORTANTE:
        - Primero, busca vuelos disponibles usando search_flights
        - Recomienda la mejor opción basándote en precio y horarios
        - Cuando vayas a reservar, usa book_flight y/o book_hotel
        - Sé claro sobre qué acciones requieren aprobación
        
        Sé amable, eficiente y transparente sobre los costos.""",
        tools=[search_flights, book_flight, book_hotel]
    ) as agent:
        
        # Solicitud del usuario
        user_request = """Necesito viajar de Madrid a Barcelona el 2025-12-15 para una reunión de negocios.
        Busca vuelos disponibles y si encuentras algo por menos de 400 EUR, por favor resérvalo a nombre de Carlos Rodríguez.
        También necesitaré hotel por 2 noches."""
        
        print(f"👤 Solicitud del usuario:\n{user_request}\n")
        print("-" * 70 + "\n")
        
        # Primera ejecución - el agente buscará vuelos y propondrá reservas
        print("🔄 Ejecutando agente (Paso 1: Búsqueda y propuesta)...\n")
        response = await agent.run([user_request])
        
        print(f"🤖 Respuesta del agente:\n{response.content}\n")
        
        # Verificar si hay funciones pendientes de aprobación
        if response.pending_approval_tool_calls:
            print("\n" + "="*70)
            print("⚠️  APROBACIÓN HUMANA REQUERIDA")
            print("="*70 + "\n")
            
            for i, tool_call in enumerate(response.pending_approval_tool_calls, 1):
                print(f"🔔 Solicitud de aprobación #{i}:")
                print(f"   Función: {tool_call.tool_name}")
                print(f"   Argumentos: {tool_call.arguments}")
                print()
                
                # Simulación de aprobación humana
                print("   👨‍💼 Gerente revisando...")
                approval_decision = "approved"  # En producción, esto vendría de un humano real
                print(f"   ✅ Decisión: {approval_decision.upper()}")
                print()
            
            # Continuar con las aprobaciones
            print("-" * 70)
            print("🔄 Ejecutando agente (Paso 2: Procesando aprobaciones)...\n")
            
            # Aprobar todas las llamadas pendientes
            for tool_call in response.pending_approval_tool_calls:
                tool_call.approve()
            
            # Continuar la ejecución con las aprobaciones
            final_response = await agent.run([])
            
            print(f"🤖 Respuesta final del agente:\n{final_response.content}\n")
        
        print("\n" + "="*70)
        print("✅ Proceso de Human-in-the-Loop completado")
        print("="*70 + "\n")


# Ejecutar el ejemplo
print("Iniciando demostración de Human-in-the-Loop...")
await run_travel_agent_with_hitl()
print("\n✓ Demostración completada")

### Ejemplo adicional: Rechazar una aprobación

Veamos qué sucede cuando un humano **rechaza** una solicitud de aprobación.

In [ ]:
async def run_travel_agent_with_rejection():
    """
    Demuestra el rechazo de una solicitud de aprobación.
    """
    
    print("\n" + "="*70)
    print("🤖 EJEMPLO: RECHAZAR APROBACIÓN")
    print("="*70 + "\n")
    
    client = OpenAIChatClient(
        model_id=MODEL_NAME,
        api_key=os.environ["GITHUB_TOKEN"],
        base_url="https://models.github.ai/inference"
    )
    
    async with ChatAgent(
        chat_client=client,
        name="travel_agent",
        instructions="""Eres un agente de viajes corporativos.
        Si una reserva es rechazada, sugiere alternativas más económicas o pregunta por requisitos modificados.""",
        tools=[search_flights, book_flight, book_hotel]
    ) as agent:
        
        user_request = "Reserva el vuelo más caro disponible de Madrid a Barcelona para Juan Pérez."
        
        print(f"👤 Solicitud: {user_request}\n")
        print("-" * 70 + "\n")
        
        response = await agent.run([user_request])
        print(f"🤖 Respuesta del agente:\n{response.content}\n")
        
        if response.pending_approval_tool_calls:
            print("\n⚠️  Aprobación requerida:")
            for tool_call in response.pending_approval_tool_calls:
                print(f"   Función: {tool_call.tool_name}")
                print(f"   Argumentos: {tool_call.arguments}")
                print()
                
                # Rechazar la solicitud
                print("   👨‍💼 Gerente de finanzas revisando...")
                print("   ❌ Decisión: RECHAZADO (fuera de presupuesto)\n")
                
                tool_call.reject("El vuelo excede el presupuesto aprobado. Por favor busca opciones más económicas.")
            
            print("-" * 70)
            print("🔄 Agente procesando el rechazo...\n")
            
            final_response = await agent.run([])
            print(f"🤖 Respuesta del agente tras rechazo:\n{final_response.content}\n")
        
        print("="*70)
        print("✅ Ejemplo de rechazo completado")
        print("="*70 + "\n")


# Ejecutar ejemplo de rechazo
await run_travel_agent_with_rejection()
print("✓ Todos los ejemplos completados")

## Resumen

### Conceptos clave aprendidos:

1. **Human-in-the-Loop (HITL)**: Patrón donde la IA colabora con humanos en decisiones críticas

2. **Aprobación de funciones**: Uso de `approval_mode="always_require"` en el decorador `@ai_function`

3. **Flujo de trabajo**:
   - Agente ejecuta y propone acciones
   - Sistema pausa cuando encuentra función que requiere aprobación
   - Humano revisa y aprueba/rechaza
   - Agente continúa basándose en la decisión

4. **Casos de uso prácticos**:
   - ✈️ Reservas de viajes corporativos
   - 💰 Transacciones financieras
   - 🏥 Decisiones médicas
   - ⚙️ Cambios de configuración críticos
   - 📧 Envío de comunicaciones importantes

### Beneficios del patrón HITL:
- ✅ Control y supervisión humana sobre decisiones críticas
- ✅ Reduce riesgos de errores costosos
- ✅ Cumplimiento de políticas y regulaciones
- ✅ Balance entre automatización y juicio humano
- ✅ Transparencia en el proceso de toma de decisiones